# 01 — Facade Segmentation

This notebook replicates the **facade segmentation** ML module from the `building_analyzer` repository.

It covers:
1. Installing dependencies
2. Cloning the repository and importing the module
3. Understanding the dataset interface
4. Instantiating and inspecting the model architectures
5. Running a training loop on a synthetic dataset
6. Running inference and visualising results
7. Evaluating with mIoU and pixel accuracy metrics


## 1. Install dependencies

In [ ]:
# Install required packages
!pip install torch torchvision albumentations Pillow numpy matplotlib --quiet
!pip install tensorboard --quiet

## 2. Clone repository and set up paths

In [ ]:
import subprocess, sys, os

# Clone the repository (update URL / branch as needed)
if not os.path.exists('building_analyzer'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Tripoid/building_analyzer.git'],
        check=True
    )

# Add the repo root to sys.path
REPO_ROOT = os.path.abspath('building_analyzer')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repository root:', REPO_ROOT)

## 3. Imports

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from ml.common.registry import ModelRegistry
from ml.common.metrics import compute_iou, compute_pixel_accuracy
from ml.common.transforms import get_segmentation_transforms
from ml.facade_segmentation.dataset import FACADE_CLASS_NAMES
from ml.facade_segmentation.model import DeepLabV3PlusSegmentation, UNetSegmentation
from ml.facade_segmentation.inference import SegmentationInferencer, SegmentationInferencerConfig
from ml.facade_segmentation.train import SegmentationTrainer, SegmentationTrainerConfig
from ml.facade_segmentation.utils import (
    colorize_mask, overlay_mask_on_image,
    compute_class_statistics, extract_class_masks
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'Classes ({len(FACADE_CLASS_NAMES)}): {FACADE_CLASS_NAMES}')

## 4. Model architecture overview

In [ ]:
NUM_CLASSES = len(FACADE_CLASS_NAMES)

# List registered models
print('Registered segmentation models:', ModelRegistry.list_models(namespace='segmentation'))

# Instantiate DeepLabV3+
deeplabv3plus = DeepLabV3PlusSegmentation(
    num_classes=NUM_CLASSES,
    class_names=FACADE_CLASS_NAMES,
    encoder_name='simple',
)
total_params = sum(p.numel() for p in deeplabv3plus.parameters())
print(f'DeepLabV3+ parameters: {total_params:,}')

# Instantiate UNet (lighter model)
unet = UNetSegmentation(
    num_classes=NUM_CLASSES,
    class_names=FACADE_CLASS_NAMES,
    base_channels=32,
)
total_params_unet = sum(p.numel() for p in unet.parameters())
print(f'UNet parameters: {total_params_unet:,}')

## 5. Synthetic dataset for demo training

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SyntheticSegDataset(Dataset):
    """Random images with random segmentation masks for quick experimentation."""
    def __init__(self, n_samples=64, image_size=(128, 128), num_classes=7):
        self.n = n_samples
        self.h, self.w = image_size
        self.num_classes = num_classes
        self.transform = get_segmentation_transforms(image_size=image_size, is_train=True)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        image = np.random.randint(0, 255, (self.h, self.w, 3), dtype=np.uint8)
        mask  = np.random.randint(0, self.num_classes, (self.h, self.w), dtype=np.int64)
        out = self.transform(image=image, mask=mask)
        return out['image'], out['mask'].long()

train_ds = SyntheticSegDataset(n_samples=128)
val_ds   = SyntheticSegDataset(n_samples=32)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False)

img, msk = train_ds[0]
print(f'Image tensor shape: {img.shape}  Mask tensor shape: {msk.shape}')

## 6. Training loop (short demo)

In [ ]:
model = UNetSegmentation(
    num_classes=NUM_CLASSES,
    class_names=FACADE_CLASS_NAMES,
    base_channels=16,
)

config = SegmentationTrainerConfig(
    output_dir='/tmp/seg_checkpoints',
    num_epochs=3,
    learning_rate=1e-3,
    device=DEVICE,
    mixed_precision=(DEVICE == 'cuda'),
    val_every_n_epochs=1,
    save_every_n_epochs=3,
    log_every_n_steps=5,
)

trainer = SegmentationTrainer(model, config)
history = trainer.train(train_loader, val_loader)

print('\nTraining complete!')
print('Train losses:', [f"{v:.4f}" for v in history['train_loss']])
print('Val mIoU:    ', [f"{v:.4f}" for v in history['val_miou']])

## 7. Learning curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], marker='o', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_miou'], marker='o', color='orange', label='Val mIoU')
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Inference and visualisation

In [ ]:
inferencer_cfg = SegmentationInferencerConfig(
    device=DEVICE,
    image_size=(128, 128),
)
inferencer = SegmentationInferencer(model=model, config=inferencer_cfg)

# Create a random test image
test_image = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
prediction = inferencer.predict_from_array(test_image)

print('Predicted mask shape:', prediction.mask.shape)
print('Class area fractions:', {k: f"{v:.3f}" for k, v in prediction.class_area_fractions().items()})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_image)
axes[0].set_title('Input Image')
axes[1].imshow(prediction.colored_mask)
axes[1].set_title('Segmentation Mask')
overlay = prediction.overlay_on(test_image, alpha=0.5)
axes[2].imshow(overlay)
axes[2].set_title('Overlay')
for ax in axes:
    ax.axis('off')
plt.suptitle('Facade Segmentation Results', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Evaluation metrics

In [ ]:
# Compute metrics on the validation set
model.eval()
all_pred, all_gt = [], []
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        preds = torch.argmax(logits, dim=1).cpu()
        all_pred.append(preds)
        all_gt.append(masks)

pred_cat = torch.cat([p.flatten() for p in all_pred])
gt_cat   = torch.cat([g.flatten() for g in all_gt])

iou_metrics = compute_iou(pred_cat, gt_cat, num_classes=NUM_CLASSES)
pixel_acc   = compute_pixel_accuracy(pred_cat, gt_cat)

print(f'Mean IoU:       {iou_metrics["mean_iou"]:.4f}')
print(f'Pixel Accuracy: {pixel_acc:.4f}')
print('Per-class IoU:')
for cls, iou in zip(FACADE_CLASS_NAMES, iou_metrics['per_class_iou']):
    print(f'  {cls:12s}: {iou:.4f}')

## 10. Swapping the model

Thanks to the registry pattern, swapping the model requires only one line change:

In [ ]:
# Swap from UNet to DeepLabV3+
model_v2 = ModelRegistry.build(
    'deeplabv3plus',
    namespace='segmentation',
    num_classes=NUM_CLASSES,
    class_names=FACADE_CLASS_NAMES,
    encoder_name='simple',
)
inferencer_v2 = SegmentationInferencer(
    model=model_v2,
    config=SegmentationInferencerConfig(device=DEVICE, image_size=(128, 128)),
)
pred_v2 = inferencer_v2.predict_from_array(test_image)
print('DeepLabV3+ mask shape:', pred_v2.mask.shape)
print('This is identical API — only the model internals differ.')